# Notebook 02: Structural Preparation — TALE-DNA Crystal Structures

**Paper C5:** *Geometric Constraints on Catalytic Domain Fusion to TALE Arrays*

This notebook documents Step 2 of the C5 pipeline: downloading, cleaning, and
annotating the priority TALE-DNA crystal structures from the RCSB PDB.

## Priority Structures

| PDB ID | TALE | Resolution |Å | Notes |
|---|---|---|---|
| 3V6T | dHax3 (holo) | 1.85 | **Primary reference** |
| 3UGM | PthXo1 | 3.0 | Complete natural TALE (22.5 repeats) |
| 3V6P | dHax3 (apo) | 2.40 | DNA-free; confirms helical pitch |
| 4HPZ | dTale2 | 2.20 | Extended N-terminal domain |
| 4OSK | Hax3 mutants | 2.40 | Non-canonical RVD recognition |
| 4GG4 | dHax3 | 2.50 | DNA-RNA hybrid, A-form geometry |
| 6LEW | AvrBs3 | 2.70 | Non-oryzae TALE (cross-species) |
| 6JTQ | Designer TALE | 2.50 | Engineered TALE |


In [ ]:
import sys
sys.path.insert(0, '../src')

import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

from tale_linker_design.structures import PRIORITY_STRUCTURES, load_reference

ANN_DIR = Path('../data/pdb_annotations')
CLEAN_DIR = Path('../data/pdb_cleaned')

print('TALE-DNA Priority Structures:')
print(f'{"PDB ID":8} {"TALE":20} {"Res (A)":8} {"Repeats":8} {"Notes"}')
print('-'*80)
for pdb_id, info in PRIORITY_STRUCTURES.items():
    print(f'{pdb_id:8} {info["tale"]:20} {info["resolution"]:8.2f} {info["repeats"]:8.1f} {info["notes"][:50]}')

In [ ]:
# Load and report all priority structures
structures = {}
for pdb_id in PRIORITY_STRUCTURES:
    try:
        tale = load_reference(pdb_id, cleaned_dir=CLEAN_DIR)
        structures[pdb_id] = tale
        src = tale.metadata.get('source', 'literature')
        n_phos = len(tale.dna_phosphate_coords)
        print(f'  {pdb_id}: TALE chain={tale.tale_chain_id}, DNA chains={tale.dna_chain_ids}, '
              f'phosphates={n_phos}, source={src}')
    except Exception as e:
        print(f'  {pdb_id}: ERROR - {e}')

In [ ]:
# Load annotation JSONs and build summary table
import pandas as pd

rows = []
for pdb_id in PRIORITY_STRUCTURES:
    ann_path = ANN_DIR / f'{pdb_id}.json'
    if ann_path.exists():
        ann = json.loads(ann_path.read_text())
        rows.append({
            'PDB ID': pdb_id,
            'TALE chain': ann['tale_chain'],
            'DNA chains': ', '.join(ann['dna_strands']),
            'C-term residue': ann['tale_c_terminus_residue'],
            'n_repeats': ann['n_repeats'],
            'Resolution (A)': ann['resolution'],
        })

ann_df = pd.DataFrame(rows)
print('Structural annotations summary:')
ann_df

In [ ]:
# Visualise TALE repeat counts vs. resolution
primary = structures.get('3V6T')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter: resolution vs repeats
res_vals   = [PRIORITY_STRUCTURES[p]['resolution'] for p in structures]
rep_vals   = [PRIORITY_STRUCTURES[p]['repeats']    for p in structures]
pdb_labels = list(structures.keys())

sc = axes[0].scatter(rep_vals, res_vals, c=res_vals, cmap='RdYlGn_r',
                     s=120, zorder=3, edgecolors='k', linewidths=0.5)
for l, x, y in zip(pdb_labels, rep_vals, res_vals):
    axes[0].annotate(l, (x, y), textcoords='offset points', xytext=(5, 4), fontsize=8)
axes[0].invert_yaxis()
axes[0].set_xlabel('Number of TALE Repeats', fontsize=12)
axes[0].set_ylabel('Resolution (AÅ)', fontsize=12)
axes[0].set_title('Resolution vs. Repeat Count\nfor Priority Structures', fontsize=12, fontweight='bold')
plt.colorbar(sc, ax=axes[0], label='Resolution (AÅ)')

# Bar: number of DNA phosphates per structure
phos_counts = {pid: len(s.dna_phosphate_coords) for pid, s in structures.items()}
colors = ['#2196F3' if pid == '3V6T' else '#90CAF9' for pid in phos_counts]
axes[1].bar(list(phos_counts.keys()), list(phos_counts.values()), color=colors, edgecolor='k', linewidth=0.5)
axes[1].set_xlabel('PDB ID', fontsize=12)
axes[1].set_ylabel('Phosphate Positions Extracted', fontsize=12)
axes[1].set_title('DNA Phosphate Coverage\nby Structure', fontsize=12, fontweight='bold')
axes[1].axhline(22, ls='--', color='#E74C3C', label='Required for bp_range=11')
axes[1].legend()

plt.tight_layout()
plt.savefig('../figures/supp_structural_prep.png', dpi=150, bbox_inches='tight')
plt.show()

## Cleaning Protocol

For each structure:
1. **Format detection:** Auto-detect mmCIF vs PDB format (Biopython 1.85+ downloads mmCIF by default).
2. **Alternate conformations:** Removed; highest-occupancy conformation retained.
3. **Solvent/buffer removal:** HOH, SO4, PO4, GOL, EDO, PEG, ACT, MPD, MES removed.
4. **Chain identification:** TALE chain = largest protein chain; DNA chains = chains with >50% nucleotide residues.
5. **Phosphate extraction:** Canonical (strand, bp_offset) mapping applied.

Script: `python scripts/clean_pdbs.py`